# PySpark drills 4: reading and writing data

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-drills-04-reading-and-writing-data.ipynb)

Practice for [PySpark 4: reading and writing data](pyspark-04-reading-and-writing-data.ipynb). The data cells write small files into `WORK_DIR`; your answer reads them.

Write your answer in the empty cell under each exercise, run it, and compare your output with the expected output.
The reference answer is hidden under the exercise; open it when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

The first cell starts a local Spark session. The second creates `orders` (one row per order; order 12 lands just after midnight UTC): a slice of the orders platform that every
module shares, introduced in [PySpark 1](pyspark-01-how-spark-runs.ipynb#setup-and-sample-data).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import functions as F
import json

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

## Exercise 1: Read a European CSV

**Difficulty:** Easy · about 10 min

Read the file at `csv_dir`: semicolon-separated, with a header, dates as `dd/MM/yyyy`, and `-` meaning missing.
Return `customer_id` (int), `signup` (date) and `segment`, ordered by `customer_id`.

**Expected output**

| customer_id | signup | segment |
| --- | --- | --- |
| 1 | 2026-01-03 | retail |
| 2 | 2026-02-15 | null |
| 3 | 2026-03-01 | wholesale |

In [ ]:
# Sample data for this exercise.
csv_dir = os.path.join(WORK_DIR, "drill_csv")
os.makedirs(csv_dir, exist_ok=True)
with open(os.path.join(csv_dir, "customers.csv"), "w") as handle:
    handle.write("customer_id;signup;segment\n1;03/01/2026;retail\n2;15/02/2026;-\n3;01/03/2026;wholesale\n")

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = (
    spark.read.schema("customer_id int, signup date, segment string")
    .option("header", True).option("sep", ";").option("nullValue", "-").option("dateFormat", "dd/MM/yyyy")
    .csv(csv_dir)
    .orderBy("customer_id")
)
result.show()

## Exercise 2: Split valid and corrupt JSON records

**Difficulty:** Medium · about 15 min

Read the JSON lines at `json_dir` with the schema `order_id int, amount decimal(12,2)` in `PERMISSIVE` mode, keeping
the raw text of bad records. Return `valid` (`order_id`, `amount` of parsed rows) and `corrupt` (the
`_corrupt_record` text of bad rows), each ordered by its first column.

**Expected output**

| order_id | amount |
| --- | --- |
| 1 | 10.50 |
| 3 | 7.00 |

| _corrupt_record |
| --- |
| {"order_id": 2, "amount": "ten"} |

In [ ]:
# Sample data for this exercise.
json_dir = os.path.join(WORK_DIR, "drill_json")
os.makedirs(json_dir, exist_ok=True)
with open(os.path.join(json_dir, "orders.json"), "w") as handle:
    handle.write('{"order_id": 1, "amount": 10.5}\n{"order_id": 2, "amount": "ten"}\n{"order_id": 3, "amount": 7}\n')

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
parsed = (
    spark.read.schema("order_id int, amount decimal(12,2), _corrupt_record string")
    .option("mode", "PERMISSIVE").option("columnNameOfCorruptRecord", "_corrupt_record")
    .json(json_dir)
    .cache()
)
valid = parsed.where("_corrupt_record IS NULL").select("order_id", "amount").orderBy("order_id")
corrupt = parsed.where("_corrupt_record IS NOT NULL").select("_corrupt_record").orderBy("_corrupt_record")
valid.show()
corrupt.show(truncate=False)

## Exercise 3: Write by date and read one day back

**Difficulty:** Medium · about 10 min

Write `orders` as Parquet partitioned by `order_date` (the date of `order_ts`) to `by_day`, then read back only
12 January 2026. Return `order_id`, `status` and `amount`, ordered by `order_id`.

**Expected output**

| order_id | status | amount |
| --- | --- | --- |
| 9 | shipped | 1180.00 |
| 10 | paid | 74.00 |

In [ ]:
# Sample data for this exercise.
by_day = os.path.join(WORK_DIR, "drill_orders_by_day")

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
orders.withColumn("order_date", F.to_date("order_ts")).write.mode("overwrite").partitionBy("order_date").parquet(by_day)
result = (
    spark.read.parquet(by_day)
    .where("order_date = DATE'2026-01-12'")
    .select("order_id", "status", "amount")
    .orderBy("order_id")
)
result.show()

## Exercise 4: Tag rows with their source file

**Difficulty:** Easy · about 10 min

Read the JSON files in `deliveries` (schema `order_id int`) and add `source_file`, the name of the file each row came
from. Order by `order_id`.

**Expected output**

| order_id | source_file |
| --- | --- |
| 1 | day1.json |
| 2 | day1.json |
| 3 | day2.json |

In [ ]:
# Sample data for this exercise.
deliveries = os.path.join(WORK_DIR, "drill_deliveries")
os.makedirs(deliveries, exist_ok=True)
for name, ids in [("day1.json", [1, 2]), ("day2.json", [3])]:
    with open(os.path.join(deliveries, name), "w") as handle:
        handle.write("\n".join(json.dumps({"order_id": i}) for i in ids))

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = (
    spark.read.schema("order_id int").json(deliveries)
    .select("order_id", F.col("_metadata.file_name").alias("source_file"))
    .orderBy("order_id")
)
result.show()

## Exercise 5: Read two schema versions together

**Difficulty:** Medium · about 10 min

`versions` holds two Parquet folders, `v=1` without a `currency` column and `v=2` with one. Read both so every column
appears, and return `v`, `currency` and `orders` (a row count), ordered by `v`.

**Expected output**

| v | currency | orders |
| --- | --- | --- |
| 1 | null | 12 |
| 2 | EUR | 3 |

In [ ]:
# Sample data for this exercise.
versions = os.path.join(WORK_DIR, "drill_versions")
orders.select("order_id").write.mode("overwrite").parquet(os.path.join(versions, "v=1"))
orders.select("order_id", F.lit("EUR").alias("currency")).where("order_id <= 3").write.mode("overwrite").parquet(os.path.join(versions, "v=2"))

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = (
    spark.read.option("mergeSchema", True).parquet(versions)
    .groupBy("v", "currency").agg(F.count("*").alias("orders"))
    .orderBy("v")
)
result.show()

**Next:** [PySpark 5: Spark SQL and the catalog](pyspark-05-spark-sql-and-the-catalog.ipynb)